In [2]:
#!/usr/bin/env python3
"""
ENGINE_Y4_independent_verification.py  — INDEPENDENT verification of the SU(3) O(y^4)
C-odd flat-band breaking result, written from scratch (does not import the
production engine). Reproduces, from primary/archived data only:

  G1  engine calibration : 3G-contraction reproduces all 2798 Stage-3F amplitudes
                           (run separately; see y4_stage3h_nonresonant_corner.gate1)
  G2  folded formula      : des-Cloizeaux H4 identity == brute-force 4th-order PT
  G3  Hermiticity         : the archived 189-entry H4 kernel is exactly Hermitian
  G4  momentum verdict     : parity-momentum corrections match the published theorem;
                           dispersion witness c4(pi,pi,pi)-c4(pi,0,0) != 0  -> LIFT
  G5  real-space verdict   : H4 * psi_cube leaks (max 5/48 on 6 plaquettes) -> LIFT
  --  Wp,Wc parity weights + demonstration that the 2-param closed form is NOT
      valid off the high-symmetry sublattice.

Usage:  python3 ENGINE_Y4_independent_verification.py  /path/to/DATA_Y4_full_real_space_h4_kernel.json.gz
"""
import sys, gzip, json, random
import sympy as sp
from fractions import Fraction as F
from collections import defaultdict

# KPATH = sys.argv[1] if len(sys.argv)>1 else 'DATA_Y4_full_real_space_h4_kernel.json.gz'
# Directly setting KPATH to the default value, as sys.argv[1] might be an unexpected value like '-f' in Colab.
KPATH = 'DATA_Y4_full_real_space_h4_kernel.json.gz'

recs = json.load(gzip.open(KPATH))['kernel']
PL = {(0,1):0,(0,2):1,(1,2):2}                 # 0=xy 1=xz 2=yz
W = defaultdict(dict)
for r in recs:
    W[(PL[tuple(r['input_plane'])],PL[tuple(r['output_plane'])])][tuple(r['displacement'])]=F(r['weight'])
print(f"loaded {len(recs)} kernel records")

# ---- G3 exact Hermiticity  T(r;a->b)=T(-r;b->a) ----
herm = all(W[(b,a)].get((-r[0],-r[1],-r[2]))==w for (a,b),d in W.items() for r,w in d.items())
print("G3 exact Hermiticity:", herm); assert herm

# ---- symbolic Bloch symbol + flat eigenvector ----
kx,ky,kz=sp.symbols('kx ky kz',real=True)
def expk(r): return sp.exp(sp.I*(r[0]*kx+r[1]*ky+r[2]*kz))
H=sp.zeros(3,3)
for (a,b),d in W.items():
    H[b,a]+=sum(sp.Rational(w.numerator,w.denominator)*expk(r) for r,w in d.items())
psi=sp.Matrix([sp.exp(sp.I*kz)-1,-(sp.exp(sp.I*ky)-1),sp.exp(sp.I*kx)-1])  # cube boundary state
def c4_at(vals):
    sub={kx:vals[0],ky:vals[1],kz:vals[2]}; Hn=H.subs(sub); pn=psi.subs(sub)
    return sp.nsimplify(sp.simplify((pn.H*Hn*pn)[0]/(pn.H*pn)[0]))
pi=sp.pi
one=c4_at((pi,0,0)); two=c4_at((pi,pi,0)); three=c4_at((pi,pi,pi))
THEO=dict(one=sp.Rational(-17700498622147435111,7250590288602460800),
          two=sp.Rational(-4367164159624988707,1812647572150615200),
          three=sp.Rational(-3447362930970494909,1450118057720492160))
print("G4 parity match:", one==THEO['one'], two==THEO['two'], three==THEO['three'])
assert one==THEO['one'] and two==THEO['two'] and three==THEO['three']
witness=sp.nsimplify(three-one)
print("   dispersion witness c4(pi,pi,pi)-c4(pi,0,0) =",witness,"=",float(witness))
assert witness==sp.Rational(17607806155349,275331901291200) and witness!=0
print("   -> band LIFTS at O(y^4)")

# ---- parity-extracted geometry weights (closed-form on the parity sublattice) ----
Wp=sp.nsimplify(one/12); Wc=sp.nsimplify((three+4*Wp)/16)
print("Wp =",Wp); print("Wc =",Wc); print("Wp-Wc =",sp.nsimplify(Wp-Wc),"!= 0  -> LIFT")

# ---- demonstrate closed form FAILS off the parity sublattice ----
def closed(vals):
    a=[2-2*sp.cos(x) for x in vals]; e1=sum(a); e2=a[0]*a[1]+a[0]*a[2]+a[1]*a[2]
    return sp.nsimplify(12*Wp-4*(Wp-Wc)*e2/e1)
kgen=(pi/2,0,0); ker=c4_at(kgen); cf=closed(kgen)
print("closed-form check at (pi/2,0,0): kernel=",ker," closed=",cf," disagree by",sp.nsimplify(ker-cf))
print("   => closed form valid only on parity sublattice; true c4(k) is richer.")

# ---- G5 real-space cube residual ----
psi_c={(2,(1,0,0)):F(1),(2,(0,0,0)):F(-1),(1,(0,1,0)):F(-1),(1,(0,0,0)):F(1),(0,(0,0,1)):F(1),(0,(0,0,0)):F(-1)}
img=defaultdict(lambda:F(0))
for (a,R),c in psi_c.items():
    for (aa,b),d in W.items():
        if aa==a:
            for r,w in d.items(): img[(b,(R[0]+r[0],R[1]+r[1],R[2]+r[2]))]+=c*w
img={k:v for k,v in img.items() if v}
c4r={img[f]/c for f,c in psi_c.items()}; assert len(c4r)==1
resid={k:(img[k]-(c4r.copy().pop()*psi_c[k] if k in psi_c else 0)) for k in img}; resid={k:v for k,v in resid.items() if v}
mx=max(abs(v) for v in resid.values())
print("G5 cube image",len(img),"| residual",len(resid),"| max leakage",mx,"on",sum(1 for v in resid.values() if abs(v)==mx),"plaquettes")
assert len(img)==36 and len(resid)==30 and mx==F(5,48)

# ---- G2 folded des-Cloizeaux formula vs brute-force PT ----
def folded_ok(seed):
    random.seed(seed); n=5; E0=F(0); diag=[E0]+[F(random.randint(2,6)) for _ in range(n-1)]
    V=sp.Matrix(n,n,lambda i,j:random.randint(-3,3)); V=V+V.T; y=sp.symbols('y')
    P=sp.zeros(n,n); P[0,0]=1; Q=sp.eye(n)-P; a=(P*V*P)[0,0]
    Rd=sp.zeros(n,n)
    for i in range(1,n): Rd[i,i]=1/(E0-diag[i])
    R=Q*Rd*Q; s=lambda X:(P*X*P)[0,0]
    H4f=s(V*R*V*R*V*R*V)-a*(s(V*R*R*V*R*V)+s(V*R*V*R*R*V))+a*a*s(V*R*R*R*V)-sp.Rational(1,2)*(s(V*R*V)*s(V*R*R*V)+s(V*R*R*V)*s(V*R*V))
    M=sp.diag(*[int(d) for d in diag])+y*V; lam=sp.symbols('lam'); cp=M.charpoly(lam).as_expr(); cks=[E0]
    for o in range(1,5):
        ck=sp.symbols('ck'); ls=sum(cks[i]*y**i for i in range(len(cks)))+ck*y**o
        cks.append(sp.nsimplify(sp.solve(sp.series(cp.subs(lam,ls),y,0,o+1).removeO().coeff(y,o),ck)[0]))
    return sp.simplify(cks[4]-sp.nsimplify(H4f))==0
g2=all(folded_ok(s) for s in (1,2,3,7,11))
print("G2 folded formula == brute-force PT (5 seeds):",g2); assert g2
print("\nALL INDEPENDENT GATES PASS — verdict: C-odd flat band LIFTS at O(y^4).")

FileNotFoundError: [Errno 2] No such file or directory: 'DATA_Y4_full_real_space_h4_kernel.json.gz'

In [4]:
#!/usr/bin/env python3
"""
ENGINE_Y4_independent_verification.py — INDEPENDENT verification of the SU(3) O(y^4)
C-odd flat-band breaking result, written from scratch (does NOT import the
production engine). Reproduces from the archived 189-entry H4 kernel only:

  G2 folded formula : des-Cloizeaux H4 identity == brute-force 4th-order PT
  G3 Hermiticity    : the 189-entry H4 kernel is exactly Hermitian
  G4 momentum verdict: parity-momentum corrections match the published theorem;
                       dispersion witness c4(pi,pi,pi)-c4(pi,0,0) != 0  -> LIFT
  G5 real-space     : H4 * psi_cube leaks (max 5/48 on 6 plaquettes) -> LIFT
  + Wp,Wc parity weights, and demonstration that the 2-param closed form is
    valid ONLY on the high-symmetry sublattice (off by 5/24 at (pi/2,0,0)).

NOTEBOOK-SAFE: auto-finds the kernel; or set  KPATH = '...'  in a cell first.
(G1 engine calibration -- 3G contraction == all 2798 Stage-3F amplitudes -- is
 checked by the production engine y4_stage3h_nonresonant_corner.gate1, not here.)
"""
import random
import sympy as sp
from fractions import Fraction as F
from collections import defaultdict

# ---- notebook-safe kernel path resolution ----
import sys, os, glob, gzip, json
import zipfile

# Extract the archive if it exists and the kernel file is not yet found
archive_path = '/content/Y4_FINAL_ARCHIVE_2026-06-13.zip'
kernel_filename = 'DATA_Y4_full_real_space_h4_kernel.json.gz'

if os.path.exists(archive_path) and not os.path.exists(kernel_filename):
    print(f"Extracting {archive_path}...")
    with zipfile.ZipFile(archive_path, 'r') as zip_ref:
        zip_ref.extractall('/content') # Extract to /content, as this is a common location checked by _find_kernel()
    print("Extraction complete.")

KPATH = globals().get("KPATH", None)   # set this in a cell to override, e.g. '/content/...json.gz'
def _find_kernel():
    fn = kernel_filename
    cands = [a for a in sys.argv[1:] if a.endswith(".gz") and os.path.exists(a)]
    for r in [os.getcwd(), ".", "/content", "/tmp/work", "/tmp/y4/Y4_STAGE3H"]:
        cands.append(os.path.join(r, fn))
    cands += glob.glob(os.path.join(os.getcwd(), "**", fn), recursive=True)
    cands += glob.glob(os.path.join("/content", "**", fn), recursive=True)
    for c in cands:
        if c and os.path.exists(c):
            return c
    raise FileNotFoundError("kernel not found; set KPATH = '/path/to/"+fn+"' in a cell first.")
if not (KPATH and os.path.exists(KPATH)):
    KPATH = _find_kernel()
print("using kernel:", KPATH)
recs = json.load(gzip.open(KPATH))["kernel"]
print(f"loaded {len(recs)} kernel records")

PL = {(0,1):0,(0,2):1,(1,2):2}          # 0=xy 1=xz 2=yz
W = defaultdict(dict)
for r in recs:
    W[(PL[tuple(r['input_plane'])],PL[tuple(r['output_plane'])])][tuple(r['displacement'])]=F(r['weight'])

# ---- G3 exact Hermiticity ----
herm = all(W[(b,a)].get((-r[0],-r[1],-r[2]))==w for (a,b),d in W.items() for r,w in d.items())
print("G3 exact Hermiticity:", herm); assert herm

# ---- symbolic Bloch symbol + flat (cube-boundary) eigenvector ----
kx,ky,kz=sp.symbols('kx ky kz',real=True)
def expk(r): return sp.exp(sp.I*(r[0]*kx+r[1]*ky+r[2]*kz))
H=sp.zeros(3,3)
for (a,b),d in W.items():
    H[b,a]+=sum(sp.Rational(w.numerator,w.denominator)*expk(r) for r,w in d.items())
psi=sp.Matrix([sp.exp(sp.I*kz)-1,-(sp.exp(sp.I*ky)-1),sp.exp(sp.I*kx)-1])
def c4_at(v):
    s={kx:v[0],ky:v[1],kz:v[2]}; Hn=H.subs(s); pn=psi.subs(s)
    return sp.nsimplify(sp.simplify((pn.H*Hn*pn)[0]/(pn.H*pn)[0]))
pi=sp.pi
one,two,three=c4_at((pi,0,0)),c4_at((pi,pi,0)),c4_at((pi,pi,pi))
THEO=dict(one=sp.Rational(-17700498622147435111,7250590288602460800),
          two=sp.Rational(-4367164159624988707,1812647572150615200),
          three=sp.Rational(-3447362930970494909,1450118057720492160))
print("G4 parity match:", one==THEO['one'], two==THEO['two'], three==THEO['three'])
assert one==THEO['one'] and two==THEO['two'] and three==THEO['three']
witness=sp.nsimplify(three-one)
print("   dispersion witness c4(pi,pi,pi)-c4(pi,0,0) =",witness,"=",float(witness))
assert witness==sp.Rational(17607806155349,275331901291200) and witness!=0
print("   -> band LIFTS at O(y^4)")

# ---- parity-extracted geometry weights ----
Wp=sp.nsimplify(one/12); Wc=sp.nsimplify((three+4*Wp)/16)
print("Wp =",Wp); print("Wc =",Wc); print("Wp-Wc =",sp.nsimplify(Wp-Wc),"!= 0 -> LIFT")

# ---- closed form fails off the parity sublattice ----
def closed(v):
    a=[2-2*sp.cos(x) for x in v]; e1=sum(a); e2=a[0]*a[1]+a[0]*a[2]+a[1]*a[2]
    return sp.nsimplify(12*Wp-4*(Wp-Wc)*e2/e1)
g=(pi/2,0,0)
print("closed-form @(pi/2,0,0): kernel - closed =",sp.nsimplify(c4_at(g)-closed(g)),
      "(both have e2/e1=0) -> closed form holds only on parity sublattice")

# ---- G5 real-space cube residual ----
psi_c={(2,(1,0,0)):F(1),(2,(0,0,0)):F(-1),(1,(0,1,0)):F(-1),(1,(0,0,0)):F(1),(0,(0,0,1)):F(1),(0,(0,0,0)):F(-1)}
img=defaultdict(lambda:F(0))
for (a,R),c in psi_c.items():
    for (aa,b),d in W.items():
        if aa==a:
            for r,w in d.items(): img[(b,(R[0]+r[0],R[1]+r[1],R[2]+r[2]))]+=c*w
img={k:v for k,v in img.items() if v}
cset={img[f]/c for f,c in psi_c.items()}; assert len(cset)==1
c4r=cset.pop()
resid={k:(img[k]-(c4r*psi_c[k] if k in psi_c else 0)) for k in img}; resid={k:v for k,v in resid.items() if v}
mx=max(abs(v) for v in resid.values())
print("G5 cube image",len(img),"| residual",len(resid),"| max leakage",mx,"on",
      sum(1 for v in resid.values() if abs(v)==mx),"plaquettes | rigid c4=",c4r)
assert len(img)==36 and len(resid)==30 and mx==F(5,48)

# ---- G2 folded des-Cloizeaux formula vs brute-force PT ----
def folded_ok(seed):
    random.seed(seed); n=5; E0=F(0); diag=[E0]+[F(random.randint(2,6)) for _ in range(n-1)]
    V=sp.Matrix(n,n,lambda i,j:random.randint(-3,3)); V=V+V.T; y=sp.symbols('y')
    P=sp.zeros(n,n); P[0,0]=1; Q=sp.eye(n)-P; a=(P*V*P)[0,0]
    Rd=sp.zeros(n,n)
    for i in range(1,n): Rd[i,i]=1/(E0-diag[i])
    R=Q*Rd*Q; s=lambda X:(P*X*P)[0,0]
    H4f=(s(V*R*V*R*V*R*V)-a*(s(V*R*R*V*R*V)+s(V*R*V*R*R*V))+a*a*s(V*R*R*R*V)
         -sp.Rational(1,2)*(s(V*R*V)*s(V*R*R*V)+s(V*R*R*V)*s(V*R*V)))
    M=sp.diag(*[int(d) for d in diag])+y*V; lam=sp.symbols('lam'); cp=M.charpoly(lam).as_expr(); cks=[E0]
    for o in range(1,5):
        ck=sp.symbols('ck'); ls=sum(cks[i]*y**i for i in range(len(cks)))+ck*y**o
        cks.append(sp.nsimplify(sp.solve(sp.series(cp.subs(lam,ls),y,0,o+1).removeO().coeff(y,o),ck)[0]))
    return sp.simplify(cks[4]-sp.nsimplify(H4f))==0
g2=all(folded_ok(s) for s in (1,2,3))   # 3 seeds (fast); add more for extra assurance
print("G2 folded formula == brute-force PT:",g2); assert g2
print("\nALL INDEPENDENT GATES PASS -> C-odd flat band LIFTS at O(y^4).")

Extracting /content/Y4_FINAL_ARCHIVE_2026-06-13.zip...
Extraction complete.
using kernel: /content/Y4_FINAL_ARCHIVE_2026-06-13/certificates/DATA_Y4_full_real_space_h4_kernel.json.gz
loaded 189 kernel records
G3 exact Hermiticity: True
G4 parity match: True True True
   dispersion witness c4(pi,pi,pi)-c4(pi,0,0) = 17607806155349/275331901291200 = 0.06395120243159332
   -> band LIFTS at O(y^4)
Wp = -17700498622147435111/87007083463229529600
Wc = -34705471293352429373/174014166926459059200
Wp-Wc = -17607806155349/4405310420659200 != 0 -> LIFT
closed-form @(pi/2,0,0): kernel - closed = -5/24 (both have e2/e1=0) -> closed form holds only on parity sublattice
G5 cube image 36 | residual 30 | max leakage 5/48 on 6 plaquettes | rigid c4= -4555981615057344457/1812647572150615200
G2 folded formula == brute-force PT: True

ALL INDEPENDENT GATES PASS -> C-odd flat band LIFTS at O(y^4).


In [5]:
#!/usr/bin/env python3
"""
ENGINE_Y4_independent_verification.py — INDEPENDENT verification of the SU(3) O(y^4)
C-odd flat-band breaking result, written from scratch (does NOT import the
production engine). Reproduces from the archived 189-entry H4 kernel only:

  G2 folded formula : des-Cloizeaux H4 identity == brute-force 4th-order PT
  G3 Hermiticity    : the 189-entry H4 kernel is exactly Hermitian
  G4 momentum verdict: parity-momentum corrections match the published theorem;
                       dispersion witness c4(pi,pi,pi)-c4(pi,0,0) != 0  -> LIFT
  G5 real-space     : H4 * psi_cube leaks (max 5/48 on 6 plaquettes) -> LIFT
  + Wp,Wc parity weights, and demonstration that the 2-param closed form is
    valid ONLY on the high-symmetry sublattice (off by 5/24 at (pi/2,0,0)).

NOTEBOOK-SAFE: auto-finds the kernel; or set  KPATH = '...'  in a cell first.
(G1 engine calibration -- 3G contraction == all 2798 Stage-3F amplitudes -- is
 checked by the production engine y4_stage3h_nonresonant_corner.gate1, not here.)
"""
import random
import sympy as sp
from fractions import Fraction as F
from collections import defaultdict

# ---- notebook-safe kernel path resolution ----
import sys, os, glob, gzip, json
KPATH = globals().get("KPATH", None)   # set this in a cell to override, e.g. '/content/...json.gz'
def _find_kernel():
    fn = "DATA_Y4_full_real_space_h4_kernel.json.gz"
    cands = [a for a in sys.argv[1:] if a.endswith(".gz") and os.path.exists(a)]
    for r in [os.getcwd(), ".", "/content", "/tmp/work", "/tmp/y4/Y4_STAGE3H"]:
        cands.append(os.path.join(r, fn))
    cands += glob.glob(os.path.join(os.getcwd(), "**", fn), recursive=True)
    cands += glob.glob(os.path.join("/content", "**", fn), recursive=True)
    for c in cands:
        if c and os.path.exists(c):
            return c
    raise FileNotFoundError("kernel not found; set KPATH = '/path/to/"+fn+"' in a cell first.")
if not (KPATH and os.path.exists(KPATH)):
    KPATH = _find_kernel()
print("using kernel:", KPATH)
recs = json.load(gzip.open(KPATH))["kernel"]
print(f"loaded {len(recs)} kernel records")

PL = {(0,1):0,(0,2):1,(1,2):2}          # 0=xy 1=xz 2=yz
W = defaultdict(dict)
for r in recs:
    W[(PL[tuple(r['input_plane'])],PL[tuple(r['output_plane'])])][tuple(r['displacement'])]=F(r['weight'])

# ---- G3 exact Hermiticity ----
herm = all(W[(b,a)].get((-r[0],-r[1],-r[2]))==w for (a,b),d in W.items() for r,w in d.items())
print("G3 exact Hermiticity:", herm); assert herm

# ---- symbolic Bloch symbol + flat (cube-boundary) eigenvector ----
kx,ky,kz=sp.symbols('kx ky kz',real=True)
def expk(r): return sp.exp(sp.I*(r[0]*kx+r[1]*ky+r[2]*kz))
H=sp.zeros(3,3)
for (a,b),d in W.items():
    H[b,a]+=sum(sp.Rational(w.numerator,w.denominator)*expk(r) for r,w in d.items())
psi=sp.Matrix([sp.exp(sp.I*kz)-1,-(sp.exp(sp.I*ky)-1),sp.exp(sp.I*kx)-1])
def c4_at(v):
    s={kx:v[0],ky:v[1],kz:v[2]}; Hn=H.subs(s); pn=psi.subs(s)
    return sp.nsimplify(sp.simplify((pn.H*Hn*pn)[0]/(pn.H*pn)[0]))
pi=sp.pi
one,two,three=c4_at((pi,0,0)),c4_at((pi,pi,0)),c4_at((pi,pi,pi))
THEO=dict(one=sp.Rational(-17700498622147435111,7250590288602460800),
          two=sp.Rational(-4367164159624988707,1812647572150615200),
          three=sp.Rational(-3447362930970494909,1450118057720492160))
print("G4 parity match:", one==THEO['one'], two==THEO['two'], three==THEO['three'])
assert one==THEO['one'] and two==THEO['two'] and three==THEO['three']
witness=sp.nsimplify(three-one)
print("   dispersion witness c4(pi,pi,pi)-c4(pi,0,0) =",witness,"=",float(witness))
assert witness==sp.Rational(17607806155349,275331901291200) and witness!=0
print("   -> band LIFTS at O(y^4)")

# ---- parity-extracted geometry weights ----
Wp=sp.nsimplify(one/12); Wc=sp.nsimplify((three+4*Wp)/16)
print("Wp =",Wp); print("Wc =",Wc); print("Wp-Wc =",sp.nsimplify(Wp-Wc),"!= 0 -> LIFT")

# ---- closed form fails off the parity sublattice ----
def closed(v):
    a=[2-2*sp.cos(x) for x in v]; e1=sum(a); e2=a[0]*a[1]+a[0]*a[2]+a[1]*a[2]
    return sp.nsimplify(12*Wp-4*(Wp-Wc)*e2/e1)
g=(pi/2,0,0)
print("closed-form @(pi/2,0,0): kernel - closed =",sp.nsimplify(c4_at(g)-closed(g)),
      "(both have e2/e1=0) -> closed form holds only on parity sublattice")

# ---- G5 real-space cube residual ----
psi_c={(2,(1,0,0)):F(1),(2,(0,0,0)):F(-1),(1,(0,1,0)):F(-1),(1,(0,0,0)):F(1),(0,(0,0,1)):F(1),(0,(0,0,0)):F(-1)}
img=defaultdict(lambda:F(0))
for (a,R),c in psi_c.items():
    for (aa,b),d in W.items():
        if aa==a:
            for r,w in d.items(): img[(b,(R[0]+r[0],R[1]+r[1],R[2]+r[2]))]+=c*w
img={k:v for k,v in img.items() if v}
cset={img[f]/c for f,c in psi_c.items()}; assert len(cset)==1
c4r=cset.pop()
resid={k:(img[k]-(c4r*psi_c[k] if k in psi_c else 0)) for k in img}; resid={k:v for k,v in resid.items() if v}
mx=max(abs(v) for v in resid.values())
print("G5 cube image",len(img),"| residual",len(resid),"| max leakage",mx,"on",
      sum(1 for v in resid.values() if abs(v)==mx),"plaquettes | rigid c4=",c4r)
assert len(img)==36 and len(resid)==30 and mx==F(5,48)

# ---- G2 folded des-Cloizeaux formula vs brute-force PT ----
def folded_ok(seed):
    random.seed(seed); n=5; E0=F(0); diag=[E0]+[F(random.randint(2,6)) for _ in range(n-1)]
    V=sp.Matrix(n,n,lambda i,j:random.randint(-3,3)); V=V+V.T; y=sp.symbols('y')
    P=sp.zeros(n,n); P[0,0]=1; Q=sp.eye(n)-P; a=(P*V*P)[0,0]
    Rd=sp.zeros(n,n)
    for i in range(1,n): Rd[i,i]=1/(E0-diag[i])
    R=Q*Rd*Q; s=lambda X:(P*X*P)[0,0]
    H4f=(s(V*R*V*R*V*R*V)-a*(s(V*R*R*V*R*V)+s(V*R*V*R*R*V))+a*a*s(V*R*R*R*V)
         -sp.Rational(1,2)*(s(V*R*V)*s(V*R*R*V)+s(V*R*R*V)*s(V*R*V)))
    M=sp.diag(*[int(d) for d in diag])+y*V; lam=sp.symbols('lam'); cp=M.charpoly(lam).as_expr(); cks=[E0]
    for o in range(1,5):
        ck=sp.symbols('ck'); ls=sum(cks[i]*y**i for i in range(len(cks)))+ck*y**o
        cks.append(sp.nsimplify(sp.solve(sp.series(cp.subs(lam,ls),y,0,o+1).removeO().coeff(y,o),ck)[0]))
    return sp.simplify(cks[4]-sp.nsimplify(H4f))==0
g2=all(folded_ok(s) for s in (1,2,3))   # 3 seeds (fast); add more for extra assurance
print("G2 folded formula == brute-force PT:",g2); assert g2
print("\nALL INDEPENDENT GATES PASS -> C-odd flat band LIFTS at O(y^4).")

using kernel: /content/Y4_FINAL_ARCHIVE_2026-06-13/certificates/DATA_Y4_full_real_space_h4_kernel.json.gz
loaded 189 kernel records
G3 exact Hermiticity: True
G4 parity match: True True True
   dispersion witness c4(pi,pi,pi)-c4(pi,0,0) = 17607806155349/275331901291200 = 0.06395120243159332
   -> band LIFTS at O(y^4)
Wp = -17700498622147435111/87007083463229529600
Wc = -34705471293352429373/174014166926459059200
Wp-Wc = -17607806155349/4405310420659200 != 0 -> LIFT
closed-form @(pi/2,0,0): kernel - closed = -5/24 (both have e2/e1=0) -> closed form holds only on parity sublattice
G5 cube image 36 | residual 30 | max leakage 5/48 on 6 plaquettes | rigid c4= -4555981615057344457/1812647572150615200
G2 folded formula == brute-force PT: True

ALL INDEPENDENT GATES PASS -> C-odd flat band LIFTS at O(y^4).


In [6]:
import base64
import gzip

PAYLOAD = r"""
H4sIACKiLWoC/609a3fbNrLf9Suw7LknZCLJkiznoa56N22cxpvEyXGcdnsdLUOJkMRIIlmSciS7+u93ZvAg+JDstPV2Iz6AwQCYFwaD4Xf/OFqnydE4CI94
eM3ibTaPwuPGd2z4d/4BvIRPomueuF7ouxOeZMF062777nS9XLorL03b8bbxHZR7F/LWeBlNFuynaOmNVb0te8REtWDiZUEUsmmUsGzO2YeP9rHDLrv/vX3U
2rF39va/fQfgTLzlZL2kom12HrGVF669JeN+kKUA9Pd1kHC/TU3+OvcyFmTMj3g6gHvGum32NlqHUPLnKJotOXuRBNecfZ3zkCXrMAzCGQtCgWGbavTa7AP3
ksmcp+xoEoUZD7OmqNZk0Gd2tAqzI9/LPI34NAgBoQ+ZN+Ot438zDzvnTbJUADxus194At0FgFh6EUZfQwYj4QeTjP37w7tzBDDjSZwEABnKLf1UYDgF1H0B
pd9mbyIPXqihTri3dNPYm3D3Vd9d8CTky/aXFAZpdsPWKXbMYyl0Y+W1pku+CcbQ+9hLUp4IiCdtdgFTEqZZsgZkCTe+Abxp6HsOS4NZyH0WL73f1zzLeMvz
v0Bz4WTLfoR5ncNMZEmwEdAet9n7JPrCEdKrvr1wGAxdREAluCnMIePBjIeENY3lKgiDVXADI4OTuYy+8jQjcPBHRa+95ZozpBsC9WMSLJfRGmbsJgq5aPkJ
Thh0zMvUpIu/JJgFPpv0m8YzP0hjnqRASkdL7i1gwtjXIAt5mpqlcHwlMq0xojnps0nEp0CxAVCDaPZpm/2aBNAmTeHRWy9Z+DixCU/XS+gM1vPY/529zwl/
DLO55IJUL6E36SQJ4oyFHLsHhIw9MEikhCH+zul1PN+mwDxLhsh9DfxsDgCBFpH7yoj+vdzfmCbRirlAgdk64a7LglUcJRl0NowyYtG00VDPYCrl5ewmiNX1
3Evny2CsbmEEkyyKlql6gCSsroHA5uo60iUSrq7S+ToLlvpuq4tkwUoXgranwZIL1CfRcgk0iogq3H0+9WDGkBtFmWniFUu8lA/E6xiQgg6ol+8RR3qRbWMS
J+L583DbZGfQOw8YTw9KuF7FWxgaFsaNRpZsBw0iOKyeToJ4247ijDhCgVEc0uCbCQdiOaUfQEbUVK/ZEERjCO18x1oH/yRteeFkHiXpHYXhr/H6/N2v5+4v
pxcvzn66dD+8et47eQytWR4/6R5PjjuPJx6fPOnyZ08fP5tOnzydTJ9NHnf9ztTvct97Nn7ymHc60yePvYnX553HnWf+k2ed6fGJJSFfnP189gIAqkG2W/2T
k5NnT7uPuyedkyfH/X7/5EmTdZ92e4/7T06e9OAxvOp1Oo6E8OvZ5fnphw8mjO6Tx50nT7HcyXH/WZP1npwcH3efdbq9Z12j5tvn/3HfnD5//fznU7P2SZP1
nzqNxruPly/OLuANzrFtKW1w9FvfffnxzRuo/uGD+9PpxaXlsGBaKmU5bRC6aZbaDuPLlNPr9uSrD/dHzKrCkM21Vws/SGwQaAAkHV4ma9A7BMmNFnQLmL2/
OH15enFx+sJ9fXpxfvrGPX/+9hRH4JaowrqPkrCae8vOa8rujEYvTt+/u7isaTQIfR5z+CfM3GtSekLHA3CkZgJntJuiyjz+4pICdaVKVIV2jcvT/1y68H9q
wwK7wmoyS76Gi5VPP9kmo1/gnnBMV5P0mn633mopLuB31/jw+uy9CyNsoNyeBVTZdePtxANl6bo5LBfuJ4s4Aq2c4tMw8rm7ivz1kqeiHayhupOCJGvB6KHI
ptega7LCg2tQZ1SPLqB/L8/Ofz69eH9xdk5dvBKAashftVHDBPpVgeLV0xLdH5hyTR53TY96v/I2wWq9cslkcKWuUi8n6zF3x2i9eMkWmkkDH2w2N4zCG55E
UGrUaDRA8LIZ6Dw79FZ8wMAOabJoMWBjUAdNEMuZFyzpMYobYLHWDyTjhOAD3LJ1im/eAwdZyH/RQjCa9fL52RuLSpFJZU+tW1F80E937JaaO6ZL0cjOcqg0
wAA1hjg0tBHhBQDxeZqixRqFp0kSJQgPYewGJgDRn3TugXy0UUcMiOMJbeiDADkHhKUCbMuiomnS7VgLFAAPbSsZQ4dBTUxzVNDaFPY0mD+oNu2ltxr73oBN
2zCRvt3t9PrsIcMfp8nGMGR5ZWq8vY59HG+CItpNOCjyEF7N+cYHgyvNbNWVlbcdI72D9ez6aP7apSmA4bJmZFW3J2g/WzR6gBwo4rbkE2MkqSW61XpP6z4T
jNbL2KYuRnhURTEVOnq7JftcTqMxlVRLy+HicFDVNpUoAwQuhdEGrkg4vR++9ICwBPSyHsZZgmc5bEVz5+8uT+VyQyCfLoI45v7RFCiG+0A8UC2nnCSKSFfA
EC8B3Svs6EhA9UIk9KsR3XzHXoJlBeYpGIBH0TqL15moC6uIJM3a7EwuZ6RBH4O5A1a8WpSwAGyPKFl5y+VWwgN1A2WWWwbsCoaoGgq0KE2V9+ES1OTxv9sN
RYwxzvWV7recGbU6spxm+VUV2P5CB6qX5rxYTmhY8Ww0MOkhzvUx2uaxolYY3SJhwIO2F6Mas+MCl3ho3dJkffWWCxetytTGsRes3gSW2YinAwCM1Hrc6bgd
sDdwVpUtaM7sRFJ1Rw+qHxMkHwVMCmRIv4hmlLaxVWrPoGRR8GowQgrxBQgs7suKU7iVHdX6b1SQKdMQX4qGigMhsHs0ZN3CY4Ap3vxgdLhQosTv6m+LS1sx
m9RNNISmoZKctOh29eJZsEK2jmHEcoZoCuZARXRFCgPM7NFIjqYg5IHBPznbpJMoQaarq26U4rDqHsJPJuVynPApT6Cmu5yg2bBp47owAeRw6DY4cntssT9Y
vb20a0i2ew+KgLPOQC66BfKkA9I2LA+BT73rKPBxJUlsHKYoSqT/AigB2wYWl9Am0WoFwmiCML/OecKJ/Q17jAl7DBh8yafA3CQ5iLIUY+WcjTSGb6RMKuog
JJQ/0e8ihcQwmNTKEcEr01e8R2bns1ziUPNPTLUqcGvhkFpkSEBpsL/oPTwAg6gDt/MAzLsBu7KEJSPAu1RptMuhkzGUY20ZQsyUMlRsD/Z/2+ipERQ41Qzh
PYbxXkP514dT2ZB6PIUd5F1zaW9CN7xwa8dt7ITmLhih+/PaLldHokdO3ojys9W2MhzetQyphwxDa/YA1YnZWD7YcuxSsGfsBd8OhcXGNgNmtzZXcuhAqMG1
GCDHKRlMDOty30aRJFGAQRdwi5KkO0BP0zohMYFCmUgtnXghOvo8n54xyespi0LQ+Ni73/pHr/qtZbBQokSqHtQaysMJNaBtKXyg7Xid3lNa0NiV1KWhKIew
LCENWSTQgoWoB0SQPJAd2LtptESLtEiy9T6SehhOWeYkhCpqgGo1HLUgXBeZDIu2Pd+3k1iStfrDEXSByKCtIsU1Koyllb+20tETqhWSac8qoIClqZYG9QKQ
9DaMbOUtNqC42coBKYqxnG+TuIQWwJQcBkbOCjHU2BIJyYdXFdDAfrgkntO/kg/xMoNVQRBGeIl+Yxc9nXgDCMwQyQKgkVM7VoDUgbHp3TEyBAJ49muU+FZp
ftP1dBpsaHbFZe38BqmLjk8XBQmUVSPSBvAprvdsS7tX9naAZIst24MRzF0iMKxmAzUSvpaJCH3hMATkYVVsO/Dj4qNKWclO7z7QsvcgrG7n4cPu0zodRAX+
OWRPO3JtKn/qwe3FWUIzu7y/oF5RY1mxoo6BeBL09fBwEvlBOBta62zaeopPsHfp0ApmIYrj8rp731/GN0j0cvHdRylGkmxvRXROHAYrQcYE0sU7+250ayHe
LQlLbVpWbQlaIJBgNJ1VB6doGptaHMGr+8OdN6XWwYImm07LohcpDgEN/rRZSL+5IWPoX3ziOLtSi7jTOeFsHAG9CVNer2HIKmBjrrbyUKmykMNyPIvqdcp3
pCvF5l0coaYOcKXOll4yk/q3Xe4vit1N1XwKC+ZTeJf5tDGMHJI6n8ojWNPOPQ0oE3Z1YsZA7Qtly7zgPiz50G3MxUykbLxlYlG84Jx2WNAimQezOU8zUUYM
yRjvYY2205YJtStXfoZ/hl9DMSzdnsGs5rZXwVuApWANhu4ulLW5tQbLXnyp74sdQrBXGiQuLTfG+hOND0FP1DztboLJhF27v3F4H8NQuJRQiuAsuGT7Fn2S
sPLVbjxyPJoTa+ooVE95H8tilSamXrLWCVKJO+m9ZQRyc1rok36eEuQDctBBv9mdm12CH3HXG+nm7s2udxdn5V2NzRaEAC5sOl110VRXNlw68nq17oRr8UI6
0G/gpgtvuj110VRXNlw68nq17oZr8UJUvNngagre9DrqoqmubLh05PXmRl50enTR2Ml5p11+N0pwD5i2X2ygKZhvmvcgzPS8BzAswLsgvuxNk9lh3Ia3fMaT
JhZzjGkPYEDQrbkpMEkHFXuA//RqpznY0xA6YZrCt+MIYQO2J8BGeWJAAqZABWvdUstXnZGzk5dduCwsu7Eo8Lqev1p09NsrKD6qxw0lV3kJ4wUJWa82zBVt
BK0tYDXb8vBmLK4DvP5iOSVxQMwVJIA7QtgIzyM+6IoH+/xmNXMIUkWCIpkgoYwMsYXIyoG4soyqiBrc4k+89EJOBvY2pl/0EG2sKtYSzrcguKFhFejgEgbk
croe24n1KX2ETcH/UdluHCVl9GI63TN3lXlLxaytJHRyGUID9mb7x/bmj5vNH5sbB5vRkFcHgK3asyRax3bXEVDD9UqiDXrMB7ULkFv/+8l/VICIlIoliVi7
NKGCDYAw8TmSaYkjFDcYJarQfqirUkMFJpRmDhM4QkIVW1a/oGpR21UTis8Q0JgBbcBuN/9I8j0I0ZyKfzClhtob3yM61Gun0oPNt8saWVNvxwu502RdZz+s
KSgNsKZmTUZXh8AhDVIhAOo495ICuNMFo8wTL4sSK+dky+chrlPzx7Vyp9iTKwOUmkJ4aIIy3UCy7VKr39ZaqZ0CfBIaQmSQOYKcmhB1eLR5rqgBrym2SF+I
ICO8/crBGqMrbwXGW7b2eZ1IuY9AyYnvamHIEik44CeI7Vxu4Ao2yaSlojtu2iu4aY+uCbC2lrhpCvYph37a6RWynq4DVUbOXpJ5iFAKhkpOTUXemUQwAHwj
dsVNBpIv9vCPfHtv9qnSJ1p4YoDNGU3EdHKYypFTphYXUEaDeUNmsCirr2nyOyX/WrAq1QlWeZ1g5c2Kd0DNyVaAceooVfZacmNp/iV+aNTWvpe4OFWP6UGw
GwTYaXecv2aa3KspFNH70CcbxtlH4GD4gqYGlKwzaVfkT5Sloffgv5AwSPcilxYo956jYxI1Rk1i2ytgd5iXyYAVN9CMnboAg2XVP1IEmPSIuF9QdAxfZp4M
k4nVr2qEnIHzYJoJV6AXpsvckJlOYcljFfd2CTxiViCzDWnzCQiSRsWNdICbSsMo5Kig6k1O4P5GUnZzvzMkr7g1Km6/peKNUfFGVnQOdwfmpaY3h6ybjVPn
uSvYJsf1zhs5RjT59sqLbZp7skgGx6O7MN3PbfUt1re2MZpCSQhdU/Rg0h5NmBh9HMqRU2ugIb1gUaUxxf22dH9jje5sj5qj1g43Vmqr1JRu6S6zzuQfEVQD
1njKbnGIEZSTm3lIAC5GJSewzo7GX3ItZWy3I2M3S4w+qlNfAEDsylf7SK+kP0ZG5JQrlnhP9PISFieik5bcYqMw+dU6zdiYM4+aQ68MBaRJf9yKZx6F5cPS
fTFg12IumnARECZtUP8rDCuoYnItO7Ar7vnrmDxalssBE7sQIKZFUJ/AzzVezvvm3SvjLodFATK6ttLRMhRPPhQhLEUJqlGrSj/oITFQcYBB9lUmpzhBVEKN
nfLCPaeRbZFB6LNUUBgw3IDWheSL4SnzmuOmv2n626Z/0y6L+kNdKw4hDcbom3pUI7HlKFe3zuSqGAjBW0qwAENRw15BSXyjZUz9ohB0NkCuSOQ6+XmyT34S
1torvVclWGkysQZMrvxw1NJM3XdH+1UJqFxZqjc4wXrCyB/gaNTX2lV3RgWSe+WxfF8lozcYqYZ7tUk2EOFpknOnNIPGHnINnxYIooZZiTauyxRyTaNTSx+E
67WB5R6xGkayRwJZ8tx/z7Iobi35NcgilKtDIVcBXQe0T2dkClfopguMmu6xlppig12GIsHjUdEfq3moFJG118zJZ0FZOqJTr/lWdIkAFU06wYhln7AQ/4Ru
2ZyrM+9GeuGyT1MgCw9r/NDKIi3RDXr7SrpJlHwtB/EbsQJ4xukOW62yGnnoAqjsIKwSeFVFQJFasqrs4mC0cr3ZbOwbfcfe0tmpVsyTVkFzixNb+R7PysvU
2Q5ToK2klBVHsISiyTfSKcDXUszu5oWktpIPqkv0VT2F5Yggka1K9nS+h2NC8mSoaWZEDOudpARHKYzbXgqX3hbkLEyejz7JIcy5jrMx17dJghHTMWf/GDL7
uMmO6yz2KjOLrjLAJhneJjuMvmYCzq0GuWvmcbIA+tgpbarigHs0MF4443ZdyxScfbgI0fGV7TXZGPSJM8KtzjqnAaB1hYVKUQ2a0PRTT1Oa6SCsyJ9mKeyC
NAkusKJ1MuHCxROtyHlDDmFS2dFX6TR2xcsSBKrqVl3M6tZVIOPcS2+uBsZ/EnVUergkxN1QcRUJHzwtickxtVSI46tibVHtENoC3O/1SKP9MGR3I/kn3WhF
IHUuNWk2bU30Gt9CWoCZFEJSwL0uW94G71gEwWd4YoNJM1xZbQWPAsnW146p/vfuzMkdOXm0VJ4MnfEIqiZbdsfW3On55fPLs3fndFjG7qBHGNaQ3Sbr4W8P
F8cjlK0bMPa2N012s9GnMoIpeePq9AXgvaZ47TvcGFtlFMrV8vZqtR7l0dGFtelWa1t1eFafhtmPBhBi+ZUwDQzVV18z/1eeClzjGhzwBZAmcrl5iavv1doZ
sEfdnPBsOU70CiHsfR/ie6rfMt9vZCX1UG1IjtfB0nfFpLv6FLEyQsp9vFO95x193Stu+AMHRCSpufSxc9ukG0MqjzEQsGZ2kKzwPwchFb3kiTp4Q+dV23ES
+WugfCHxW5IMEx5zLxse14T+Am9G43shZ9re6HjUSMlTxfBoXK9gaoNKqL+/1/cXz1yNnZqARwonF6Hw49ihzRoQB8JNOP69dsEjKtUjJoRnul4BtKtlEC5G
7CEgJS9xePAKR0dAqQ9rgkYA0P44otc9UwqKTTkSegYTvO4BVRLh3EGWinjxrLstHgoztMkWA7RecATBeiH6zW8Fdq+EgYPCM7WFvaIsHHgsxXK399TRtKsR
1ytVeQazsh56JewDlD3Km4udhAENMeI7trtf4Fo4eOGJH2X2AgEX40deyf7NebIKMtBCWX5Sr9Bd6h9BEyh8jRIKtOm0O38CeXTahxztP1kCfZs21PUkv9ut
TR68kxTc9WbrK29j0zXUHKd2YSBajMY4/KIfy1ad0hiIMSIwWmKLLAZAEZO+HIb+/mkXEorASBlF11I8nSOREfm8Bu0kj9y96uunEjy9ElvQOANoZKi5w16i
nQw9PIdunYteOe1L1ZH5gRrQUguaq9RB5U/VfmBd3nqKjrS5cX/n6UeR/eFVTjcMj7StE45m9mJ4u9gN2PnwltoA2/pVf3hL8HfKVcevYTg+9gSLLDFobNZG
e8c+FwiuvHQhFwjQDSjNHrE+EAH7J+vx1hO5xg0XksOxeBtFS94/8RZodGCsuc5R7FLihKm3XI69yUIdvtEWyGztQdWMo3crwwwM6K0IOWv1jXwU5uosXVwh
mslsFYR2BWESQnhm2xhSQrsrBuIjXH7sXQ2aBEksq8QZT3iu5439C4nmX+xj4X2nfQJcLg5zQnP0m0+1HOd+eYyhA+m85gSooB+oIOIKEM2moAp15C3xYndh
3y351LaJsbxbqG15wBMex4HD/gcMNiGw8K4lLtSh4CQAsTzxwjL/hbmdVnQ3iJaxb7q/5HC0WwS3ydQPRqn5dKTbPNMpIwflnpgVhFMM6sF1q/jXKcg9u1Ut
J0pkUSz8j7kjh6QYIlaMKVpsq8/1u5v6dxQSJdmChvVqAQbXAozdxc1IjXBVk6vJdNEAqBFuWjAVFC2Ibewq1Eaiire2JIp6xQ+MSYGPnVG9bpYDjFDr6/8g
RncvAD32BkJVVGD8lScVm5LSBp/WnMDZ6DicinVsheg0Nfz1wNzwRHbReL5wjTfdUTuLaIVgbO2J4ws4ci78GqV7JhwQ2vBcDUGxAeNVfQs6C4yrp9cABrxV
xTvl3KdTWhsTZq52YciuBt3eaFS04xM+DUKOvShzJgGULk36R6jB0V5eRReRyl0iw2yNwD+Tygm0nqtvJ2k1u3ISJ8182hb5cOJqPVzAMwHZAnUwj3AcLZJG
eFrKkkc375IWSiKIM6noErZrIjvq8ZZSduM40GOpLDdZ4hW2oq7AICKriP4x5vVKirkDr/IS+1/Sj3ydb0DhPMgTUn4KgpzwMruFslfNqV2Uak3T2YcAlLgq
ej3EoA+tc770edJ6yz2/5BeJ6DRDOryl/BOwJMHI1xM6eLjxYDIxjJa3uhhXOzXue7scjFMgMznggH1782cp7JAEVMSCcksBxQQX1GK6nkx4mjqSVqpy6C/I
HUnOVVmjMgwlrmxelTouSCRN/8ZkwJJ/mQUtCrqyyqIBkza4uP70QPmK/CJyJaX8QtWdkHz/oSkhpPVHtA1jG0eP/SE5DbdPxIDLcywUQe6vV3Gqm22qTYAh
RkboswuPmAX/e1QNfjd2B2LhDheINQpHZmDxJeobLULJ+sYUJ+PJhiBzp2h6yANzqTolI5iseEzugfLY63GlrGYwLQTiQdMoeWcRi5z0QTgT0szcjtZp0cxX
I6eIdQZ27F9DGiEcxrlUwtqbmchqVtuDvhHFYiXrQaETwKKl0UdPCrpAC92rRo1iWaEOiJaO5Pltn06opEwNaYuO05o792aLNHKVBgmPSosi4dPeBik3HawG
zKZkTcERKmIiCCfLNU6pSpVjtm1JrkVDygVD2w8wMUxqH8qSILg1NdMr5NlKcj4EkIEInRCqA6HjoXp4IvNpHYlYimuOx07zxq1CcZkGq5T4ikrMo5SyNKgG
6L4NxEinUxFHp+YFpZ3Ij9KUzwddDXrKxKgkgdDnmAXEyjbhvmP2FWecONCw9whzHnZfPYVcgaVPH2MEQmPv+Umf1I+ehCN5ILm8vQbFiulRZHYiHzfa/zHU
eY3q8jGUW5hat3i0lK927q2uB+ZkZ7S7ffCg/QVWX7Y6OMtT8yCHTltQPJh6v+OjIhkgrQ16eNwTUa/xayJ1qkUCzgQWy8vdmd2nkuFnEq2XgrGxZXYb7/LM
PoWwS2xYp1aCIShlU8o6MIqYt7CN/9jKEVLJwdQwEltZQwuG49ljx3z2W59hZjn25t2vpx8uW5hgjv307vTly7Of0Nk8YBenP7375fTiN9BdmHjuDF6QB9py
akEbqVWaxjG2SroWKmfnPK5euaKoOA7otL9i0kxxpksPqqFCby3RFC5OxNHMcgqGkTqr6avDmv6uSZlOYFHfM0ygytHZhjD/CpnBRK5XkbsAJu5KxOEaDkiR
1GC0K4zO1LoA8woGIhdfUBkDfgRCjipekTLdk1E1VZQ8Hji83Vw9oKsHo0Hf3zGGD1BgPRjtxLlXKoIXD0bam+aBiiSxhzLxDjEnRKZQEVQ+zoc3h0M8+Bfy
cYzMPUbZmHms687WrVzPU9hbERk6RLXnvKJtJAjUp+udkTKwyUjVg/V3db6Q32dkNGWEo+mWTCQG+wV3bejMoXjaunDK3LYgbJTc03E1BaFLlsbdJ3rrMvLM
xCpHS/tqthpJ6b/IdCwijygm0KO6O6vWr6QAD1ldFtJ6T5EUX5gVUCdZhvIYkEI5nZHdQZD6mP65ptX9h/QlYBL6JmA/mE7BeBYxtyb075nK5RyAzZZm+C9G
r679AF00f0XpiPkk6mAinVxR8cj4O8o7vS1lnFapXynDtIxjaN+55kGmJCOdQgpAa4FGmvQtWbaI6izKdHyHcZ4jvqqDUApzoYSQlsJV5XAGdiagihAoaWwT
LQ14vCvm/dN5nPSaRiZQ/lZk98I5iHJeXiVuLiMv89XuQ78+seY34l4P5CDiUIXpNJ5FjI08ufuwJjdnNk+i9WzuRonPE3e7jzwKzWJFSm89TqIFR8FNHaiH
N0JfoVyToaY+VLR0hk7E7cgMpG7uM6UK90KU4ld1RSYqmngcbGGknQH5UR5Gx2KFgrG2/6XYPUB99w1AJa/rVVf9iq6p01cd1MzwkNOJa8v6joEdqTOXi6Tm
jcalznZfkSaYck5YetjceJ2J1Pi4XMX1e0skfZ9G6ySbtwh7FVT01UtlsinKdt+4kDn9KWHVoNFin++Tzvhzo/Ga5NyMhxhcgQf+QQhT/ZrMk0cHldxnqKVF
+XVJZ32+rdNGu89ieMSGosjsAFYFiugt0+tdgEpdkoJNCcIcKEm2HbYv57xGoOSlpSih8pLlJQKSk/OiBg9LVFOu0IyTCPOUoF2lHBo0g35EE+NzTOAUhCKn
oJkYv5px/mOM6RuooJwI6LScaEpAJjOKlZLiIwt8vhWLh93ndsMy8tEYawq1dHHPT09fnL7AFNCFFQUScE0iipLBbX0Kf3zz7qfXpy8GJX+KQYeVWmDz5+wQ
U0j7rcbsQRWzB7saED8pzhQcmxZg1HlFClBW3oK7N0FsC3Yv73E0DOOe7EsdZIH7J4Xly6fwI6XEUHEet0Yt1eLrfiEyuxA4bhSXMkUISzmMMn5epAnVWZ/p
RMTrvsN+wGgm0CPyfqdOmKQTb+klpShD8a0EM5MlsLWcAqCt7tNnKn77e/l9C1w2I2+NkWTzdDIq3FxCTfHrFKBO8L9wiilxRbQmA5qd0/cmvFB+XwRHithq
uaX2JLp4EKBdGNaz85fvmIyblIWItrZqeYi9La0kqQ4mCw9CaT6oYHC9puSZnZB15jbxPwpEA0j1oAT/V8aoMDOpshHnYtumNgYIWjBm1hDiRuQHTCqB+Cdt
s9Csoh2hYAxv8e2gfcwNq/RnFWyR8uW0lWEiHKMdGRqaf2sEXWJESIqAenQY9/hxTkK9wnp7UXRH6yisY8M3kO+hd9rHT5qs1Wk/7TZZt909HtUWq9shq6YZ
psxCe+ItVBBQzrWiw/l3UcTHTlpoBogwEty+fcT6Mu5FDC/G1+j9HoxcD2bDW36dP1IDjQEUtP7MwymQo0MjH5TdPW6yHkYuPnOMjT7hLijkRRYUVtzUs36+
OHvBwuFtevUgfDAa9HxYgYCaoAfwC48etbsnsx2G0uAz2rUiV4ZVggQ0I2p5m7xWvvHGyOahEjWb3VCDKljGFp8ktVceSaEQpCqML5AuDrngeMqthfwOPL2K
fNJUsDybeGvcHEA+nIBU5RSMQjm7VHZM0BBL1H1KIESUhpfpbTWxB7peCaL2e1239wy3TcZojMGgXrW6oyva2cO9efmopx7VMDRhsEL/F6zmQJwOe93WD71n
uHgm4AX2Elv0PrkO9GY9zrvRttj9LzSVB6eAjn15dn76QsykBGfM5xMxnwvjnZ5XywCD+49GGbzFInKL1wBMD9Cd1ch3Z5XIwTJ0LCaMRMYvKZZxSGilIIDI
0fznkFVG+BFyzjNjE3kqwfrlzgEvIVigMg1DvrC0kW1sbmLrnsjseJ+9z6apl/Vmp96cKtRFAr0s7EZN+vRpLT2rqssN48Q4fkCKi3MMmM9PynsFWBk63+st
et/gL0P8BGluuE/6FQtPzBJf1uH8srihlSMtIiUMAxd3X7+1J0Fpt+37shENOsMXqfUOd1F1oR7XwlGrCjJFEbh3rw2Td4P5MVtHa7BKpLeGMgQWIYKGL4pC
2rtsqbnrD6EnuaBrlrYb4T0NwaNCqYoYFIs3ndHMQNw1ciCaFl2zUlB49rCocPHtKS1MH1eYPq6y4wbK8CtuOZPBUzglp4ruNXgMAHUmC1RHe8NEH5eEbtnt
RKJb9tukyjsr+nwSrKidKklXKlc9RzUNyxXcvWrXt74XQr0PyGzcWBOa1bXBQPkv8deMKxFsm9ObK/Udxk6JdyZB6FKGUHSLTJBXlLKgtn4heMGNgd/k7FSA
3UPaVAm8GImQTzlGjBXEfpU1iOVdoQ2oL3hRjmEQq2YSMuj1HCiJY5SrdKT4wCjJl15MfIbf76NAPmPjEIyKrKODc9T2mloP5mtp5VQpYKY/zMiFs79Y31xq
F8JeUMCYe3A1S3Cxn4mGyTQh/5L43OMn+9Lt/vcWP/m4++QItfPJFh9/hHvpbBBaKEeu0Wix13Lp+rmwdv2s3xj+GoEgGCqmMAPFjoXfF1ZrUmQZqzUo8mLf
4kzBrRdm0ABUNsPklZyCqnpdBEUu7nAGoeiuErU0kwV+d3uIykCU2MjBvP0WBxJgLS3NXMVKQTBgDx/WG44PH2IzEY4OFvtcNSER8EtYEMmVJkXL42qc02gX
bcl8prWRU+NWK3AvwT8rsBUALvIZwn1pfAGSaO+TPXH7nxwCKBhX+fvUVxpxSaC+1Cg/i6qPRURTOajK2Y0f+IBlrQfvcD0BPVzBYviap4367zsqr1mBHQy3
G7lIADb0A22PaIxdhhkZbxsvo3WCX/2gNEro5lAZdMXRGPKL6JW98UEDaUOp0qbjthHFIn+dMff6a5+CeXvIvKUvfuZevZVfFUfqg3gibCH/Lp4ITzhFV59Z
1xRFK3+vwNEOP4zUqw2XgMXO8zd7Qh/yZ0USKS7SKgwgAnmU0UoiTfwNWT1flNw4goofFTkLgRbNP1YvFwxld6gRoDcBe5+lr21SaEXRfAkGZuUB/tH6ofz+
7YsB3N3KKdPrqJIbExC+qligZKVowPJeAsJMlyqQRkIyQ9N0So08uOZmL7kRnWnn7Y8fz1+8OW0DRCv/LJv8gGj7/4L4JfzaNzLpMR6RRyYCSY6yd6jL4WeW
Tl++Aap9QTmQb4pnPxJPfMOhGACgvulCARxQxNn/TRwZABGk9NEG+1tS/98IvsH4KC+Z4I7/UGz8O40/l0I+BrKofvCjeCL+jpig5p7CNc7+fUXvZcvsq3wP
yZNXrcnve+AjO3cMt+KTH2kPBDjpRvNJAyC7Ln0GwqWgDNfFqDHXlfEXIoSs8f+XcpnO9nsAAA==
""".replace("\n", "").replace(" ", "")

source = gzip.decompress(base64.b64decode(PAYLOAD)).decode("utf-8")
print(f"Recovered script: {len(source):,} characters")
exec(compile(source, "recover_and_certify_y4_full_mass.py", "exec"))

Recovered script: 31,734 characters
Mounted at /content/drive
Y4 FULL LOWEST-MASS COEFFICIENT: RECOVERY + CERTIFICATION
Search roots: ['/content', '/content/drive/MyDrive']
Ranked candidates: 9
  score=2000  /content/DATA_Y4_full_real_space_h4_kernel.json.gz  hits=['exact_direct_path']
  score=1720  /content/Y4_FINAL_ARCHIVE_2026-06-13/certificates/CERT_Y4_stage3j_verdict.json  hits=['17607806155349', '1812647572150615200', '275331901291200', '4555981615057344457', 'cube_boundary_residual_nonzero', 'maximum_exact_leakage', 'name_keyword', 'preferred_filename', 'y4_full_real_space_H4_kernel']
  score= 420  /content/Y4_FINAL_ARCHIVE_2026-06-13/THM_Y4_theorem_flat_band_breaking_2026-06-13.md  hits=['17607806155349', '1812647572150615200', '275331901291200', '4555981615057344457', 'name_keyword']
  score= 420  /content/Y4_FINAL_ARCHIVE_2026-06-13/certificates/y4_cube_boundary_residual.json  hits=['17607806155349', '1812647572150615200', '4555981615057344457', 'name_keyword', 'y4_full_real_

KeyError: ['src', 'source', 'from', 'a', 'i', 'row', 'ori_from', 'source_orientation', 'orientation_from', 'p']

In [7]:
# PATCH CELL — run directly after the failed recovery cell.
# Replaces the kernel-schema parser and restarts certification.

import re
import json
from pathlib import Path
from fractions import Fraction
from collections import defaultdict

import numpy as np

_required = [
    "main",
    "read_json_any",
    "find_records",
    "parse_orientation",
    "parse_complex_exact",
]
_missing = [name for name in _required if name not in globals()]

if _missing:
    raise RuntimeError(
        "Run the original recovery cell once before this patch. "
        f"Missing definitions: {_missing}"
    )


# ---------------------------------------------------------------------------
# Flexible key classifications
# ---------------------------------------------------------------------------

SRC_WORDS = {
    "src",
    "source",
    "from",
    "bra",
    "left",
    "row",
    "initial",
    "input",
    "in",
    "origin_state",
    "source_state",
    "from_state",
    "p",
    "p1",
    "face1",
    "plaquette1",
    "source_plaquette",
    "alpha",
    "a",
    "a_state",
    "o1",
}

DST_WORDS = {
    "dst",
    "target",
    "to",
    "ket",
    "right",
    "col",
    "final",
    "output",
    "out",
    "destination",
    "target_state",
    "to_state",
    "q",
    "p2",
    "face2",
    "plaquette2",
    "target_plaquette",
    "beta",
    "b",
    "b_state",
    "o2",
}

DISP_WORDS = {
    "r",
    "dr",
    "delta",
    "disp",
    "displacement",
    "shift",
    "translation",
    "offset",
    "relative_position",
    "relative_displacement",
}

VALUE_WORDS = {
    "value",
    "val",
    "rational",
    "fraction",
    "coeff",
    "coefficient",
    "coef",
    "weight",
    "amplitude",
    "amp",
    "entry",
    "matrix_element",
    "element",
    "w",
    "c",
}

MATRIX_WORDS = {
    "matrix",
    "h4",
    "block",
    "value_matrix",
    "kernel_matrix",
    "matrix_element_block",
}


# ---------------------------------------------------------------------------
# Recursive schema utilities
# ---------------------------------------------------------------------------

def _name_matches(key, words):
    key = str(key).lower()
    tokens = [
        token
        for token in re.split(r"[^a-z0-9]+", key)
        if token
    ]

    for word in words:
        word = word.lower()

        if (
            key == word
            or word in tokens
            or key.startswith(word + "_")
            or key.endswith("_" + word)
        ):
            return True

    return False


def _recursive_candidates(obj, words, path=()):
    output = []

    if isinstance(obj, dict):
        for key, value in obj.items():
            new_path = path + (str(key),)

            if _name_matches(key, words):
                output.append((len(new_path), new_path, value))

            output.extend(
                _recursive_candidates(
                    value,
                    words,
                    new_path,
                )
            )

    elif isinstance(obj, (list, tuple)):
        for index, value in enumerate(obj):
            output.extend(
                _recursive_candidates(
                    value,
                    words,
                    path + (str(index),),
                )
            )

    return output


def _first_recursive(obj, words):
    candidates = _recursive_candidates(obj, words)

    if not candidates:
        raise KeyError(sorted(words))

    # Prefer the shallowest matching field.
    candidates.sort(key=lambda item: item[0])

    return candidates[0][2], candidates[0][1]


# ---------------------------------------------------------------------------
# Position and orientation parsing
# ---------------------------------------------------------------------------

def _parse_vec3(value, *, prefer_last=False):
    if isinstance(value, dict):

        for keys in [
            ("dx", "dy", "dz"),
            ("x", "y", "z"),
            ("i", "j", "k"),
        ]:
            if all(key in value for key in keys):
                return tuple(
                    int(value[key])
                    for key in keys
                )

        for key in [
            "r",
            "R",
            "dr",
            "delta",
            "disp",
            "displacement",
            "shift",
            "translation",
            "offset",
            "position",
            "pos",
            "origin",
            "site",
            "cell",
            "coord",
            "coords",
            "coordinate",
            "xyz",
        ]:
            if key in value:
                try:
                    return _parse_vec3(
                        value[key],
                        prefer_last=prefer_last,
                    )
                except Exception:
                    pass

    if isinstance(value, str):
        numbers = list(
            map(
                int,
                re.findall(r"-?\d+", value),
            )
        )

        if len(numbers) >= 3:
            selected = (
                numbers[-3:]
                if prefer_last
                else numbers[:3]
            )

            return tuple(selected)

    if isinstance(value, (list, tuple, np.ndarray)):
        values = list(value)

        if len(values) >= 3:
            try:
                selected = (
                    values[-3:]
                    if prefer_last
                    else values[:3]
                )

                return tuple(
                    int(item)
                    for item in selected
                )

            except Exception:
                pass

    raise ValueError(
        f"Cannot parse a three-vector from {value!r}"
    )


def _state_orientation(state):
    try:
        return parse_orientation(state)
    except Exception:
        pass

    if isinstance(state, dict):

        priority = [
            "orientation",
            "ori",
            "plane",
            "orientation_index",
            "plane_index",
            "type",
            "o",
            "mu_nu",
            "directions",
        ]

        for key in priority:
            if key in state:
                try:
                    return parse_orientation(state[key])
                except Exception:
                    pass

        for value in state.values():
            try:
                return parse_orientation(value)
            except Exception:
                pass

    if isinstance(state, (list, tuple)):
        for value in state:
            try:
                return parse_orientation(value)
            except Exception:
                pass

    raise ValueError(
        f"Cannot parse orientation from state {state!r}"
    )


def _state_position(state):
    if isinstance(state, dict):

        for key in [
            "position",
            "pos",
            "origin",
            "site",
            "cell",
            "coord",
            "coords",
            "coordinate",
            "xyz",
            "location",
        ]:
            if key in state:
                try:
                    return _parse_vec3(
                        state[key],
                        prefer_last=True,
                    )
                except Exception:
                    pass

        try:
            return _parse_vec3(
                state,
                prefer_last=True,
            )
        except Exception:
            pass

    if isinstance(state, str):
        return _parse_vec3(
            state,
            prefer_last=True,
        )

    if isinstance(state, (list, tuple)):

        # Prefer explicitly nested coordinate objects.
        for value in state:
            if isinstance(
                value,
                (list, tuple, dict, str),
            ):
                try:
                    return _parse_vec3(
                        value,
                        prefer_last=True,
                    )
                except Exception:
                    pass

        return _parse_vec3(
            state,
            prefer_last=True,
        )

    raise ValueError(
        f"Cannot parse position from state {state!r}"
    )


# ---------------------------------------------------------------------------
# Exact-value parsing
# ---------------------------------------------------------------------------

def _fraction_from_pair(record, base):
    candidate_pairs = [
        (
            f"{base}_numerator",
            f"{base}_denominator",
        ),
        (
            f"{base}_num",
            f"{base}_den",
        ),
        (
            f"{base}_n",
            f"{base}_d",
        ),
    ]

    for numerator_key, denominator_key in candidate_pairs:

        if (
            numerator_key in record
            and denominator_key in record
        ):
            return Fraction(
                int(record[numerator_key]),
                int(record[denominator_key]),
            )

    raise KeyError(base)


def _direct_value(record):
    # Complex rational:
    # real_num / real_den + i imag_num / imag_den

    for real_base in ["real", "re"]:
        for imag_base in ["imag", "im", "imaginary"]:

            try:
                real = _fraction_from_pair(
                    record,
                    real_base,
                )

                imag = _fraction_from_pair(
                    record,
                    imag_base,
                )

                return complex(
                    float(real),
                    float(imag),
                )

            except Exception:
                pass

    # Plain numerator / denominator.

    if (
        "numerator" in record
        and "denominator" in record
    ):
        return complex(
            float(
                Fraction(
                    int(record["numerator"]),
                    int(record["denominator"]),
                )
            ),
            0.0,
        )

    if (
        "num" in record
        and "den" in record
    ):
        return complex(
            float(
                Fraction(
                    int(record["num"]),
                    int(record["den"]),
                )
            ),
            0.0,
        )

    # Prefixed rational pairs.

    for base in [
        "value",
        "val",
        "weight",
        "coeff",
        "coefficient",
        "coef",
        "amplitude",
        "amp",
        "entry",
        "matrix_element",
        "w",
        "c",
    ]:
        try:
            return complex(
                float(
                    _fraction_from_pair(
                        record,
                        base,
                    )
                ),
                0.0,
            )
        except Exception:
            pass

    value, value_path = _first_recursive(
        record,
        VALUE_WORDS,
    )

    return parse_complex_exact(value)


# ---------------------------------------------------------------------------
# Compact sparse-index parsing
# ---------------------------------------------------------------------------

def _compact_index(record):
    for key in [
        "key",
        "index",
        "indices",
        "label",
        "term",
        "transition",
        "kernel_key",
        "record_key",
        "id",
    ]:
        if key not in record:
            continue

        value = record[key]

        # Schema:
        # [[mu,nu], [rho,sigma], [dx,dy,dz]]

        if (
            isinstance(value, (list, tuple))
            and len(value) >= 3
            and isinstance(value[0], (list, tuple))
            and isinstance(value[1], (list, tuple))
        ):
            source_orientation = parse_orientation(
                value[0]
            )

            target_orientation = parse_orientation(
                value[1]
            )

            displacement = _parse_vec3(
                value[2]
            )

            return (
                source_orientation,
                target_orientation,
                displacement,
            )

        numbers = list(
            map(
                int,
                re.findall(
                    r"-?\d+",
                    str(value),
                ),
            )
        )

        # Schema:
        # [orientation_index_a,
        #  orientation_index_b,
        #  dx, dy, dz]

        if len(numbers) == 5:
            return (
                numbers[0],
                numbers[1],
                tuple(numbers[2:5]),
            )

        # Schema:
        # [mu,nu,rho,sigma,dx,dy,dz]

        if len(numbers) >= 7:
            source_orientation = parse_orientation(
                numbers[0:2]
            )

            target_orientation = parse_orientation(
                numbers[2:4]
            )

            return (
                source_orientation,
                target_orientation,
                tuple(numbers[-3:]),
            )

    raise ValueError(
        "No compact sparse index found"
    )


# ---------------------------------------------------------------------------
# One-record adapter
# ---------------------------------------------------------------------------

def _record_to_entry(record):
    # List schema:
    # [a,b,dx,dy,dz,value]

    if not isinstance(record, dict):

        if (
            isinstance(record, (list, tuple))
            and len(record) >= 6
        ):
            return (
                parse_orientation(record[0]),
                parse_orientation(record[1]),
                tuple(
                    int(value)
                    for value in record[2:5]
                ),
                parse_complex_exact(record[5]),
            )

        raise TypeError(
            f"Unsupported record type: {type(record)}"
        )

    # Matrix-per-displacement schema.

    matrix = None
    matrix_key = None

    for key, value in record.items():

        if _name_matches(key, MATRIX_WORDS):

            array = np.asarray(
                value,
                dtype=object,
            )

            if array.shape == (3, 3):
                matrix = array
                matrix_key = key
                break

    if matrix is not None:

        try:
            displacement_value, displacement_path = (
                _first_recursive(
                    record,
                    DISP_WORDS,
                )
            )

            displacement = _parse_vec3(
                displacement_value
            )

        except Exception:
            displacement = (0, 0, 0)

        return (
            "MATRIX",
            displacement,
            matrix_key,
            matrix,
        )

    # Sparse compact-key schema.

    try:
        source_orientation, target_orientation, displacement = (
            _compact_index(record)
        )

    except Exception:

        source_state, source_path = _first_recursive(
            record,
            SRC_WORDS,
        )

        target_state, target_path = _first_recursive(
            record,
            DST_WORDS,
        )

        source_orientation = _state_orientation(
            source_state
        )

        target_orientation = _state_orientation(
            target_state
        )

        try:
            displacement_value, displacement_path = (
                _first_recursive(
                    record,
                    DISP_WORDS,
                )
            )

            displacement = _parse_vec3(
                displacement_value
            )

        except Exception:
            source_position = _state_position(
                source_state
            )

            target_position = _state_position(
                target_state
            )

            displacement = tuple(
                int(
                    target_position[index]
                    - source_position[index]
                )
                for index in range(3)
            )

    value = _direct_value(record)

    return (
        source_orientation,
        target_orientation,
        displacement,
        value,
    )


# ---------------------------------------------------------------------------
# Replacement kernel parser
# ---------------------------------------------------------------------------

def parse_kernel_v2(path):
    path = Path(path)

    obj = read_json_any(path)
    records, metadata = find_records(obj)

    print("\n" + "=" * 88)
    print("KERNEL SCHEMA DISCOVERY")
    print("=" * 88)

    print("kernel path:", path)
    print("top-level type:", type(obj).__name__)

    if isinstance(obj, dict):
        print(
            "top-level keys:",
            list(obj.keys())[:40],
        )

    print("record count:", len(records))

    if records:
        sample = json.dumps(
            records[0],
            indent=2,
            default=str,
        )

        print("\nFIRST RECORD")
        print(sample[:5000])

    kernel = defaultdict(complex)

    skipped = []
    parsed_records = 0

    for record_index, record in enumerate(records):

        try:
            entry = _record_to_entry(record)

            if entry[0] == "MATRIX":

                (
                    _,
                    displacement,
                    matrix_key,
                    matrix,
                ) = entry

                for source_orientation in range(3):
                    for target_orientation in range(3):

                        kernel[
                            (
                                source_orientation,
                                target_orientation,
                                tuple(displacement),
                            )
                        ] += parse_complex_exact(
                            matrix[
                                source_orientation,
                                target_orientation,
                            ]
                        )

            else:
                (
                    source_orientation,
                    target_orientation,
                    displacement,
                    value,
                ) = entry

                source_orientation = int(
                    source_orientation
                )

                target_orientation = int(
                    target_orientation
                )

                displacement = tuple(
                    int(component)
                    for component in displacement
                )

                if not (
                    0 <= source_orientation <= 2
                    and 0 <= target_orientation <= 2
                ):
                    raise ValueError(
                        "Orientation index outside 0,1,2: "
                        f"{source_orientation}, "
                        f"{target_orientation}"
                    )

                kernel[
                    (
                        source_orientation,
                        target_orientation,
                        displacement,
                    )
                ] += complex(value)

            parsed_records += 1

        except Exception as exc:

            skipped.append(
                {
                    "index": record_index,
                    "error": repr(exc),
                    "record": record,
                }
            )

    print("\n" + "=" * 88)
    print("SCHEMA PARSE RESULT")
    print("=" * 88)

    print(
        f"parsed records: "
        f"{parsed_records}/{len(records)}"
    )

    print(
        f"aggregated sparse entries: "
        f"{len(kernel)}"
    )

    print(
        f"skipped records: "
        f"{len(skipped)}"
    )

    if skipped:

        print("\nFIRST SKIPPED RECORD")

        print(
            json.dumps(
                skipped[0],
                indent=2,
                default=str,
            )[:6000]
        )

        schema_dump_path = Path(
            "/content/Y4_kernel_schema_skipped.json"
        )

        schema_dump_path.write_text(
            json.dumps(
                skipped[:20],
                indent=2,
                default=str,
            ),
            encoding="utf-8",
        )

        print(
            "\nschema dump written to:",
            schema_dump_path,
        )

    if not kernel:
        raise RuntimeError(
            "No kernel entries were parsed. "
            "Send the printed FIRST RECORD and the file "
            "/content/Y4_kernel_schema_skipped.json."
        )

    # Prevent a partial kernel from silently producing a false spectrum.

    required_count = max(
        1,
        int(0.90 * len(records)),
    )

    if parsed_records < required_count:
        raise RuntimeError(
            f"Only {parsed_records}/{len(records)} "
            "records parsed. The schema dump above "
            "identifies the remaining record format."
        )

    return dict(kernel), metadata


# Install the replacement.

parse_kernel = parse_kernel_v2

print(
    "\nParser patched successfully. "
    "Restarting full recovery and certification...\n"
)

main()


Parser patched successfully. Restarting full recovery and certification...

Y4 FULL LOWEST-MASS COEFFICIENT: RECOVERY + CERTIFICATION
Search roots: ['/content', '/content/drive/MyDrive']
Ranked candidates: 9
  score=2000  /content/DATA_Y4_full_real_space_h4_kernel.json.gz  hits=['exact_direct_path']
  score=1720  /content/Y4_FINAL_ARCHIVE_2026-06-13/certificates/CERT_Y4_stage3j_verdict.json  hits=['17607806155349', '1812647572150615200', '275331901291200', '4555981615057344457', 'cube_boundary_residual_nonzero', 'maximum_exact_leakage', 'name_keyword', 'preferred_filename', 'y4_full_real_space_H4_kernel']
  score= 420  /content/Y4_FINAL_ARCHIVE_2026-06-13/THM_Y4_theorem_flat_band_breaking_2026-06-13.md  hits=['17607806155349', '1812647572150615200', '275331901291200', '4555981615057344457', 'name_keyword']
  score= 420  /content/Y4_FINAL_ARCHIVE_2026-06-13/certificates/y4_cube_boundary_residual.json  hits=['17607806155349', '1812647572150615200', '4555981615057344457', 'name_keyword',

/usr/lib/python3.12/zipfile/__init__.py:1624: UserWarning: Duplicate name: 'DATA_Y4_full_real_space_h4_kernel.json.gz'
  return self._open_to_write(zinfo, force_zip64=force_zip64)
/usr/lib/python3.12/zipfile/__init__.py:1624: UserWarning: Duplicate name: 'y4_full_lowest_mass_certificate.json'
  return self._open_to_write(zinfo, force_zip64=force_zip64)
/usr/lib/python3.12/zipfile/__init__.py:1624: UserWarning: Duplicate name: 'Y4_FULL_LOWEST_MASS_CERTIFICATE.md'
  return self._open_to_write(zinfo, force_zip64=force_zip64)


In [9]:
# =============================================================================
# EXACT k=0 CERTIFICATE FOR THE FULL O(y^4) T1^{+-} MASS COEFFICIENT
#
# Reads the recovered 189-record exact rational kernel and:
#   1. reconstructs H4(k=0) exactly;
#   2. checks exact Hermiticity and cubic degeneracy;
#   3. extracts the exact eigenvalue corresponding to the numerical minimum;
#   4. separates the rigid shift from the band-breaking correction;
#   5. freezes kernel/verdict hashes and writes JSON + Markdown certificates.
# =============================================================================

import gzip
import hashlib
import json
import shutil
from fractions import Fraction
from pathlib import Path

import numpy as np
import sympy as sp


# ---------------------------------------------------------------------------
# Paths and known anchors
# ---------------------------------------------------------------------------

KERNEL_PATH = Path(
    "/content/DATA_Y4_full_real_space_h4_kernel.json.gz"
)

ARCHIVE = Path(
    "/content/Y4_FINAL_ARCHIVE_2026-06-13"
)

if ARCHIVE.exists():
    OUTDIR = ARCHIVE / "certificates"
else:
    OUTDIR = Path(
        "/content/Y4_EXACT_K0_CERT"
    )

OUTDIR.mkdir(
    parents=True,
    exist_ok=True,
)

VERDICT_PATH = (
    ARCHIVE
    / "certificates"
    / "CERT_Y4_stage3j_verdict.json"
)

RIGID = sp.Rational(
    -4555981615057344457,
    1812647572150615200,
)

NUMERICAL_MINIMUM = sp.Float(
    "-2.8579159881145655",
    50,
)


# ---------------------------------------------------------------------------
# Utilities
# ---------------------------------------------------------------------------

def sha256(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def plane_index(plane):
    """
    Canonical orientation ordering:

        0 = xy = {0,1}
        1 = yz = {1,2}
        2 = zx = {0,2}

    The JSON records use plane pairs, so orientation sign is irrelevant
    for the basis label here.
    """

    pair = frozenset(
        int(value)
        for value in plane
    )

    mapping = {
        frozenset((0, 1)): 0,
        frozenset((1, 2)): 1,
        frozenset((0, 2)): 2,
    }

    if pair not in mapping:
        raise ValueError(
            f"Unknown plaquette plane: {plane}"
        )

    return mapping[pair]


def exact_weight(value):
    fraction = Fraction(
        str(value)
    )

    return sp.Rational(
        fraction.numerator,
        fraction.denominator,
    )


def gate(name, condition, detail=""):
    status = (
        "PASS"
        if condition
        else "FAIL"
    )

    print(
        f"{status:4s} "
        f"{name:38s} "
        f"{detail}"
    )

    if not condition:
        raise AssertionError(
            f"{name}: {detail}"
        )


# ---------------------------------------------------------------------------
# Load the exact real-space kernel
# ---------------------------------------------------------------------------

gate(
    "kernel file exists",
    KERNEL_PATH.exists(),
    str(KERNEL_PATH),
)

with gzip.open(
    KERNEL_PATH,
    "rt",
    encoding="utf-8",
) as handle:
    payload = json.load(handle)

records = payload["kernel"]
metadata = payload.get(
    "meta",
    {},
)

gate(
    "kernel record count",
    len(records) == 189,
    str(len(records)),
)

print("\n" + "=" * 96)
print("KERNEL METADATA")
print("=" * 96)
print(
    json.dumps(
        metadata,
        indent=2,
        sort_keys=True,
    )
)


# ---------------------------------------------------------------------------
# Construct H4(0) exactly
#
# At k=0 every displacement phase is exp(i k.r)=1, so each real-space
# hopping weight contributes directly to the corresponding 3x3 entry.
# ---------------------------------------------------------------------------

H0 = sp.zeros(
    3,
    3,
)

displacement_counts = {}

for record in records:
    source = plane_index(
        record["input_plane"]
    )

    target = plane_index(
        record["output_plane"]
    )

    displacement = tuple(
        int(value)
        for value in record["displacement"]
    )

    weight = exact_weight(
        record["weight"]
    )

    H0[
        source,
        target,
    ] += weight

    displacement_counts[
        displacement
    ] = (
        displacement_counts.get(
            displacement,
            0,
        )
        + 1
    )


# ---------------------------------------------------------------------------
# Exact matrix checks
# ---------------------------------------------------------------------------

print("\n" + "=" * 96)
print("EXACT H4(k=0)")
print("=" * 96)

sp.pprint(H0)

gate(
    "exact Hermiticity",
    H0 == H0.T,
)

gate(
    "all entries rational",
    all(
        entry.is_Rational
        for entry in H0
    ),
)

trace_average = sp.factor(
    sp.trace(H0) / 3
)

scalar_residual = sp.simplify(
    H0
    - trace_average * sp.eye(3)
)

is_scalar = (
    scalar_residual
    == sp.zeros(3)
)

gate(
    "cubic scalar on T1 triplet",
    is_scalar,
    f"trace/3 = {trace_average}",
)


# ---------------------------------------------------------------------------
# Exact eigenvalue
# ---------------------------------------------------------------------------

eigenvalues = H0.eigenvals()

print("\nExact eigenvalues and multiplicities:")

for eigenvalue, multiplicity in eigenvalues.items():
    print(
        "  ",
        eigenvalue,
        " multiplicity=",
        multiplicity,
        " decimal=",
        sp.N(
            eigenvalue,
            30,
        ),
    )

gate(
    "single triply-degenerate eigenvalue",
    (
        len(eigenvalues) == 1
        and list(
            eigenvalues.values()
        )[0] == 3
    ),
    str(eigenvalues),
)

C4_EXACT = sp.factor(
    next(
        iter(
            eigenvalues.keys()
        )
    )
)

C4_DECIMAL = sp.N(
    C4_EXACT,
    50,
)

difference_from_scan = abs(
    C4_DECIMAL
    - NUMERICAL_MINIMUM
)

gate(
    "exact value matches optimized scan",
    difference_from_scan
    < sp.Float(
        "5e-13",
        50,
    ),
    f"difference={difference_from_scan}",
)


# ---------------------------------------------------------------------------
# Separate rigid and band-breaking parts
# ---------------------------------------------------------------------------

BREAKING_CORRECTION = sp.factor(
    C4_EXACT
    - RIGID
)

print("\n" + "=" * 96)
print("EXACT FOURTH-ORDER DECOMPOSITION")
print("=" * 96)

print(
    "rigid component exact       =",
    RIGID,
)

print(
    "rigid component decimal     =",
    sp.N(
        RIGID,
        30,
    ),
)

print(
    "full k=0 coefficient exact  =",
    C4_EXACT,
)

print(
    "full k=0 coefficient decimal=",
    C4_DECIMAL,
)

print(
    "breaking correction exact   =",
    BREAKING_CORRECTION,
)

print(
    "breaking correction decimal =",
    sp.N(
        BREAKING_CORRECTION,
        30,
    ),
)


# ---------------------------------------------------------------------------
# Updated mass series
# ---------------------------------------------------------------------------

A0 = sp.Rational(
    8,
    3,
)

A1 = sp.Integer(
    1
)

A2 = sp.Rational(
    11,
    306,
)

A3 = sp.Rational(
    -109151,
    249696,
)

y = sp.symbols(
    "y",
    real=True,
)

MASS_SERIES = (
    A0
    + A1 * y
    + A2 * y**2
    + A3 * y**3
    + C4_EXACT * y**4
)

print("\n" + "=" * 96)
print("UPDATED EXACT MASS SERIES")
print("=" * 96)

print(
    "m_-(y) ="
)

sp.pprint(
    MASS_SERIES
)

print(
    "\nLaTeX:"
)

print(
    sp.latex(
        MASS_SERIES
    )
)


# ---------------------------------------------------------------------------
# Freeze current provenance
# ---------------------------------------------------------------------------

kernel_hash = sha256(
    KERNEL_PATH
)

verdict_hash = (
    sha256(
        VERDICT_PATH
    )
    if VERDICT_PATH.exists()
    else None
)

result = {
    "title": (
        "Exact k=0 O(y^4) coefficient "
        "for the SU(3) T1^{+-} lowest band"
    ),
    "kernel_path": str(
        KERNEL_PATH
    ),
    "kernel_sha256": kernel_hash,
    "verdict_path": (
        str(
            VERDICT_PATH
        )
        if VERDICT_PATH.exists()
        else None
    ),
    "verdict_sha256": verdict_hash,
    "kernel_record_count": len(
        records
    ),
    "distinct_displacements": len(
        displacement_counts
    ),
    "metadata": metadata,
    "H4_k0_exact": [
        [
            str(
                H0[row, column]
            )
            for column in range(3)
        ]
        for row in range(3)
    ],
    "H4_k0_is_scalar": bool(
        is_scalar
    ),
    "c4_rigid_exact": str(
        RIGID
    ),
    "c4_rigid_decimal": float(
        RIGID
    ),
    "c4_full_k0_exact": str(
        C4_EXACT
    ),
    "c4_full_k0_decimal": float(
        C4_EXACT
    ),
    "c4_breaking_correction_exact": str(
        BREAKING_CORRECTION
    ),
    "c4_breaking_correction_decimal": float(
        BREAKING_CORRECTION
    ),
    "numerical_scan_minimum": float(
        NUMERICAL_MINIMUM
    ),
    "exact_minus_numerical": float(
        C4_DECIMAL
        - NUMERICAL_MINIMUM
    ),
    "mass_series_exact": str(
        MASS_SERIES
    ),
    "mass_series_latex": sp.latex(
        MASS_SERIES
    ),
    "status": (
        "Exact coefficient at k=0. "
        "The remaining theorem obligation is a "
        "global Brillouin-zone minimum certificate."
    ),
}

json_path = (
    OUTDIR
    / "y4_exact_k0_mass_coefficient.json"
)

json_path.write_text(
    json.dumps(
        result,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)


# ---------------------------------------------------------------------------
# Markdown certificate
# ---------------------------------------------------------------------------

markdown = f"""# Exact \(k=0\) fourth-order mass coefficient

## Result

The exact fourth-order Bloch matrix at \(k=0\) is

```text
{str(H0)}
```
"""


PASS kernel file exists                     /content/DATA_Y4_full_real_space_h4_kernel.json.gz
PASS kernel record count                    189

KERNEL METADATA
{
  "basis_planes": [
    [
      0,
      1
    ],
    [
      0,
      2
    ],
    [
      1,
      2
    ]
  ],
  "stage3i_input": "/mnt/data/Y4_FROM_SCRATCH_TEST/Y4_STAGE3I/y4_complete_folded_word_weights.json.gz",
  "stage3i_sha256": "854a02e981098de7fcfd1a14dd5c9703aff0c36a2a81ea5589ddf4ff8c321bd0",
  "version": "2026-06-13-stage3j-v1"
}

EXACT H4(k=0)
⎡-20721577909065127111                                                 ⎤
⎢──────────────────────            0                       0           ⎥
⎢ 7250590288602460800                                                  ⎥
⎢                                                                      ⎥
⎢                        -20721577909065127111                         ⎥
⎢          0             ──────────────────────            0           ⎥
⎢                         72505902886024

<>:563: SyntaxWarning: invalid escape sequence '\('
<>:563: SyntaxWarning: invalid escape sequence '\('
/tmp/ipykernel_8764/3668088221.py:563: SyntaxWarning: invalid escape sequence '\('
  """


In [10]:
# Exact k=0 certificate for the full O(y^4) T1^{+-} coefficient.
# Self-contained. Paste into ONE new Colab cell and run.

import gzip
import hashlib
import json
from fractions import Fraction
from pathlib import Path

import sympy as sp

KERNEL_PATH = Path("/content/DATA_Y4_full_real_space_h4_kernel.json.gz")
VERDICT_PATH = Path(
    "/content/Y4_FINAL_ARCHIVE_2026-06-13/"
    "certificates/CERT_Y4_stage3j_verdict.json"
)
OUTDIR = Path("/content/Y4_EXACT_K0_CERT")
OUTDIR.mkdir(parents=True, exist_ok=True)

RIGID = sp.Rational(
    -4555981615057344457,
    1812647572150615200,
)
NUMERICAL_SCAN_MIN = sp.Float("-2.8579159881145655", 60)


def gate(name, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    print(f"{status:4s} {name:38s} {detail}")
    if not condition:
        raise AssertionError(f"{name}: {detail}")


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def plane_index(plane):
    pair = frozenset(int(x) for x in plane)
    mapping = {
        frozenset((0, 1)): 0,  # xy
        frozenset((1, 2)): 1,  # yz
        frozenset((0, 2)): 2,  # zx/xz
    }
    if pair not in mapping:
        raise ValueError(f"Unknown plane: {plane}")
    return mapping[pair]


def exact_rational(value):
    frac = Fraction(str(value))
    return sp.Rational(frac.numerator, frac.denominator)


gate("kernel exists", KERNEL_PATH.exists(), str(KERNEL_PATH))

with gzip.open(KERNEL_PATH, "rt", encoding="utf-8") as handle:
    payload = json.load(handle)

records = payload["kernel"]
metadata = payload.get("meta", {})

gate("kernel record count", len(records) == 189, str(len(records)))

# At k=0, exp(i k.r) = 1 for every displacement.
H0 = sp.zeros(3, 3)
displacements = set()

for record in records:
    a = plane_index(record["input_plane"])
    b = plane_index(record["output_plane"])
    r = tuple(int(x) for x in record["displacement"])
    w = exact_rational(record["weight"])

    H0[a, b] += w
    displacements.add(r)

gate("exact Hermiticity", H0 == H0.T)

print("\nEXACT H4(k=0)")
sp.pprint(H0)

trace_average = sp.factor(sp.trace(H0) / 3)
scalar_residual = sp.simplify(H0 - trace_average * sp.eye(3))
is_scalar = scalar_residual == sp.zeros(3)

print("\nScalar on T1 triplet:", is_scalar)
print("trace(H0)/3 =", trace_average)
print("decimal     =", sp.N(trace_average, 40))

eigenvalues = H0.eigenvals()

print("\nEXACT EIGENVALUES")
for eigenvalue, multiplicity in eigenvalues.items():
    print(
        "  value =",
        eigenvalue,
        " multiplicity =",
        multiplicity,
        " decimal =",
        sp.N(eigenvalue, 40),
    )

# Select the exact k=0 lowest eigenvalue.
if is_scalar:
    C4_K0 = trace_average
else:
    C4_K0 = min(
        eigenvalues.keys(),
        key=lambda x: float(sp.N(x, 30)),
    )

C4_K0 = sp.factor(C4_K0)
BREAKING_K0 = sp.factor(C4_K0 - RIGID)
SCAN_DIFFERENCE = sp.N(C4_K0 - NUMERICAL_SCAN_MIN, 50)

print("\n" + "=" * 88)
print("EXACT k=0 RESULT")
print("=" * 88)
print("c4(k=0) exact         =", C4_K0)
print("c4(k=0) decimal       =", sp.N(C4_K0, 50))
print("rigid exact           =", RIGID)
print("rigid decimal         =", sp.N(RIGID, 50))
print("breaking at k=0 exact =", BREAKING_K0)
print("breaking decimal      =", sp.N(BREAKING_K0, 50))
print("exact - scan minimum  =", SCAN_DIFFERENCE)

gate(
    "matches numerical optimizer",
    abs(float(SCAN_DIFFERENCE)) < 5e-12,
    f"difference={SCAN_DIFFERENCE}",
)

A0 = sp.Rational(8, 3)
A1 = sp.Integer(1)
A2 = sp.Rational(11, 306)
A3 = sp.Rational(-109151, 249696)
y = sp.symbols("y", real=True)

MASS_SERIES = sp.expand(
    A0 + A1*y + A2*y**2 + A3*y**3 + C4_K0*y**4
)

kernel_hash = sha256(KERNEL_PATH)
verdict_hash = sha256(VERDICT_PATH) if VERDICT_PATH.exists() else None

result = {
    "title": "Exact k=0 O(y^4) coefficient for the SU(3) T1^{+-} band",
    "kernel_path": str(KERNEL_PATH),
    "kernel_sha256": kernel_hash,
    "verdict_path": str(VERDICT_PATH) if VERDICT_PATH.exists() else None,
    "verdict_sha256": verdict_hash,
    "kernel_record_count": len(records),
    "distinct_displacements": len(displacements),
    "metadata": metadata,
    "H4_k0_exact": [
        [str(H0[row, col]) for col in range(3)]
        for row in range(3)
    ],
    "H4_k0_is_scalar": bool(is_scalar),
    "H4_k0_eigenvalues": {
        str(value): int(mult)
        for value, mult in eigenvalues.items()
    },
    "c4_rigid_exact": str(RIGID),
    "c4_rigid_decimal": float(RIGID),
    "c4_k0_exact": str(C4_K0),
    "c4_k0_decimal": float(sp.N(C4_K0, 30)),
    "c4_breaking_at_k0_exact": str(BREAKING_K0),
    "c4_breaking_at_k0_decimal": float(sp.N(BREAKING_K0, 30)),
    "numerical_scan_minimum": float(NUMERICAL_SCAN_MIN),
    "exact_minus_scan": float(SCAN_DIFFERENCE),
    "mass_series_exact": str(MASS_SERIES),
    "mass_series_latex": sp.latex(MASS_SERIES),
    "status": (
        "Exact at k=0 and matched to the numerical Brillouin-zone optimizer. "
        "A global analytic or interval-arithmetic lower-bound certificate "
        "is still required to prove that k=0 is the global minimum."
    ),
}

json_path = OUTDIR / "y4_exact_k0_mass_coefficient.json"
json_path.write_text(
    json.dumps(result, indent=2, sort_keys=True),
    encoding="utf-8",
)

# Build Markdown line-by-line to avoid triple-quoted-string failures.
markdown_lines = [
    "# Exact k=0 fourth-order mass coefficient",
    "",
    "## Result",
    "",
    "The exact fourth-order Bloch matrix at k=0 is:",
    "",
    "```text",
    str(H0),
    "```",
    "",
    f"Scalar on the T1 triplet: `{is_scalar}`",
    "",
    "Exact k=0 coefficient:",
    "",
    "```text",
    str(C4_K0),
    "```",
    "",
    f"Decimal: `{sp.N(C4_K0, 30)}`",
    "",
    "Rigid component:",
    "",
    "```text",
    str(RIGID),
    "```",
    "",
    "Additional band-breaking contribution at k=0:",
    "",
    "```text",
    str(BREAKING_K0),
    "```",
    "",
    f"Decimal: `{sp.N(BREAKING_K0, 30)}`",
    "",
    "Updated exact mass series through fourth order:",
    "",
    "```text",
    f"m_-(y) = {MASS_SERIES} + O(y**5)",
    "```",
    "",
    "## Provenance",
    "",
    f"- Kernel SHA256: `{kernel_hash}`",
    f"- Verdict SHA256: `{verdict_hash}`",
    f"- Kernel records: `{len(records)}`",
    f"- Distinct displacements: `{len(displacements)}`",
    "",
    "## Remaining obligation",
    "",
    (
        "This certificate establishes the exact coefficient attained at k=0 "
        "and its agreement with the numerical optimizer. It does not yet prove "
        "that no other momentum has a lower eigenvalue. The final theorem step "
        "is a global analytic or interval-arithmetic Brillouin-zone bound."
    ),
    "",
]

md_path = OUTDIR / "Y4_EXACT_K0_MASS_COEFFICIENT.md"
md_path.write_text(
    "\n".join(markdown_lines),
    encoding="utf-8",
)

print("\nUPDATED EXACT SERIES")
sp.pprint(MASS_SERIES)

print("\nLaTeX:")
print(sp.latex(MASS_SERIES))

print("\n" + "=" * 88)
print("FILES WRITTEN")
print("=" * 88)
print(json_path)
print(md_path)

print("\nSTATUS")
print(
    "Exact at k=0 and numerically coincident with the Brillouin-zone minimum. "
    "The remaining proof obligation is global minimality."
)

PASS kernel exists                          /content/DATA_Y4_full_real_space_h4_kernel.json.gz
PASS kernel record count                    189
PASS exact Hermiticity                      

EXACT H4(k=0)
⎡-20721577909065127111                                                 ⎤
⎢──────────────────────            0                       0           ⎥
⎢ 7250590288602460800                                                  ⎥
⎢                                                                      ⎥
⎢                        -20721577909065127111                         ⎥
⎢          0             ──────────────────────            0           ⎥
⎢                         7250590288602460800                          ⎥
⎢                                                                      ⎥
⎢                                                -20721577909065127111 ⎥
⎢          0                       0             ──────────────────────⎥
⎣                                                 72505902886024608